In [1]:
import sys
from pathlib import Path
import gc

# Find the repo root and add it to sys.path FIRST
repo = Path.cwd()
while not (repo / "functions").exists() and repo.parent != repo:
    repo = repo.parent
sys.path[:0] = [str(repo), str(Path.cwd().parent.parent)]

import tempfile, py7zr
import geopandas as gpd
import glob, json
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from rasterio.warp import reproject, calculate_default_transform

from constants import *
from functions.plot_hillshade import plot_hillshade
from utils import *
from cs_plotting import *
from cs_data_process import *
from cs_constants import *

CS_PATH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/Ecosystems/eco_data/COStatewideConservationSummaryV8/TIF_File/ConservationSummaryV8_NoTribalLands.tif")
ELEVATION_PATH = Path("/Users/kylabazlen/Documents/Climate_Roadmap/co-risk-assessment/co_rm/ecosystems/cs/data/elevation.tif")
VEG_PATH = (
    "/Users/kylabazlen/Documents/Climate_Roadmap/Forest/Data/"
    "Vegetation_COWRA22/Vegetation_COWRA22.tif"
)

### PLOT BASE CS MAP LAYER ###

In [2]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list(
    "vivid_blues",
    [
        (0.00, "#F0F7FF"),
        (0.30, "#B8DBFF"),  # pale sky blue holds until value 3
        (0.60, "#5BA8F0"),  # light blue until value 6
        (0.80, "#1E5FE6"),  # medium blue until value 8
        (0.92, "#0827B8"),
        (1.00, "#000C66"),
    ],
    N=256,
)

In [ ]:
cs_arr, cs_extent = reproject_raster(CS_PATH, TARGET_CRS)
cs_arr = np.ma.masked_where(
    (cs_arr == -3.4028234663852886e+38) |
    np.isnan(cs_arr) |
    (cs_arr <= 8),
    cs_arr,
)

# Hillshade base
fig, ax = plot_hillshade()

# Conservation summary
cmap.set_bad(color="gray", alpha=.8)
im = ax.imshow(cs_arr, extent=cs_extent, cmap=cmap, vmin=0, vmax=10,
               interpolation="nearest", alpha=0.75, zorder=2)
fig.colorbar(im, ax=ax, shrink=0.6, label="Conservation Score (1–10)")

# Counties on top
plt_counties(COUNTIES_PATH, county_edgecolor="black", county_linewidth=1.0, ax=ax)

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

### PLOT Bivariate ###

In [ ]:
# ============================================================
# Pipeline
# ============================================================

LAYERS_TO_PLOT = ["Days Exceeding 95th Percentile Maximum Temperature"] #["drought__D3"] #["5 day max precip"]
#, "Days Exceeding 95th Percentile Maximum Temperature"
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

CS_PATH = Path(
    "/Users/kylabazlen/Documents/Climate_Roadmap/Ecosystems/eco_data/"
    "COStatewideConservationSummaryV8/TIF_File/ConservationSummaryV8_NoTribalLands.tif"
)
OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/bivariate")

# CS = fine reference grid — build once
cs_transform, cs_width, cs_height = build_reference_grid(str(CS_PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(CS_PATH), cs_transform, cs_width, cs_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(cs_transform, cs_width, cs_height)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue

    for fp in files:
        # Upsample climate to the CS grid
        clim_aligned = reproject_to_grid(
            fp, cs_transform, cs_width, cs_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        # Classify
        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        # Save everything
        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        save_bivariate_outputs(
            run_dir,
            bivar_map=codes_2d,
            extent=extent,
            color_dict=layers[layer_name].get("bivar"),
            x_label="Conservation Score →",
            y_label=f"{cfg.get('label', layer_name)} →",
            n_classes=N_CLASSES,
            x_breaks=x_breaks,
            y_breaks=y_breaks,
            meta_extra={
                "layer_name": layer_name,
                "climate_file": str(fp),
                "cs_file": str(CS_PATH),
                "x_variable": "Conservation Score (CS)",
                "y_variable": cfg.get("label", layer_name),
                "y_units": cfg.get("cbar_label", ""),
                "classification_method": "quantile",
            },
        )
        print(f"✓ saved {run_dir.name}/")

## Contours map ##

## Map with only pixels with 8 9 and 10 conservation importance ##

### PLOT VEGETATION ###

In [2]:
VEG_LABELS = {
    1: "Agriculture",
    3: "Grassland",
    5: "Lodgepole Pine",
    6: "Mixed Conifer",
    7: "Oak Shrubland",
    8: "Open Water",
    9: "Pinyon-Juniper",
    10: "Ponderosa Pine",
    11: "Riparian",
    12: "Shrubland",
    13: "Spruce-Fir",
    14: "Developed",
    15: "Sparsely Vegetated",
    16: "Hardwood",
    17: "Conifer-Hardwood",
    18: "Conifer",
    19: "Barren",
}


In [17]:
LAYERS_TO_PLOT = ["5 day max precip", "drought__D3", "Days Exceeding 95th Percentile Maximum Temperature"]
TARGET_CRS = "EPSG:3857"
N_CLASSES = 3

PATH = CS_PATH  # swap this for any conservation/priority raster
OUT_ROOT = Path("/Users/kylabazlen/Documents/Climate_Roadmap/outputs/cs_vegetation")

# Build reference grid from the conservation raster
ref_transform, ref_width, ref_height = build_reference_grid(str(PATH), dst_crs=TARGET_CRS)
cs_aligned = reproject_to_grid(
    str(PATH), ref_transform, ref_width, ref_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)
extent = grid_to_extent(ref_transform, ref_width, ref_height)

# Align vegetation to the reference grid (categorical → nearest)
veg_aligned = reproject_to_grid(
    str(VEG_PATH), ref_transform, ref_width, ref_height,
    dst_crs=TARGET_CRS, resampling=Resampling.nearest,
)

selected = LAYERS_TO_PLOT or list(layers.keys())

for layer_name in selected:
    if layer_name not in layers:
        print(f"⚠️  Unknown layer: {layer_name}")
        continue
    cfg = layers[layer_name]
    files = sorted(glob.glob(cfg["path"]))
    if not files:
        print(f"⚠️  No files for {layer_name}: {cfg['path']}")
        continue
    for fp in files:
        clim_aligned = reproject_to_grid(
            fp, ref_transform, ref_width, ref_height,
            dst_crs=TARGET_CRS, resampling=Resampling.bilinear,
        )

        (cs_valid, clim_valid), mask_2d = mask_valid_pixels(cs_aligned, clim_aligned)
        x_breaks, y_breaks, codes = bivariate_classify(
            cs_valid, clim_valid, n_classes=N_CLASSES, method="quantile"
        )
        codes_2d = unflatten_to_raster(codes, mask_2d, fill=-1)

        bivar_colors = layers[layer_name].get("bivar")
        hazard_label = cfg.get("label", layer_name)

        fig_veg, _ = plot_veg_composition(
            bivar_map=codes_2d,
            veg_aligned=veg_aligned,
            bivar_colors=bivar_colors,
            hazard_label=hazard_label,
            sort_by="size",
        )

        run_dir = OUT_ROOT / f"{layer_name}__{Path(fp).stem}"
        run_dir.mkdir(parents=True, exist_ok=True)
        fig_veg.savefig(run_dir / "veg_composition.png", dpi=150, bbox_inches="tight")

        # Clean up — order matters
        plt.close(fig_veg)
        plt.close('all')  # belt and suspenders, kills any stragglers
        del clim_aligned, cs_valid, clim_valid, mask_2d, codes, codes_2d, fig_veg
        gc.collect()

        print(f"✓ saved {run_dir.name}/veg_composition.png")

KeyboardInterrupt: 